# **Scraping de FotMob y FBref**

Se corre en Colab. Cada sección necesita correr antes las dos celdas de instalación e imports.

| Sección | Entra | Sale | Copiar a (en el proyecto) |
|---|---|---|---|
| Tiros por partido | `codigos_fotmob.xlsx` (de `data/manual/`) | `fotmob_<torneo>_partido<N>.csv` | `data/raw/fotmob/partidos/<torneo>/` |
| Posiciones de FBref | — | `info_jugadores_<torneo>.csv` | `data/raw/fbref/` |
| Equipos de FotMob | — | `id_equipos_<torneo>.csv` | `data/raw/fotmob/equipos/` |

# **Instalación y preparación de las librerías.**

In [ ]:
!pip install LanusStats==1.8.2 openpyxl

In [ ]:
import time
import pandas as pd # Importación de la librería que uso para el manejo de las bases.
import LanusStats as ls # Importación de la librería y cambio de nombre.
from google.colab import files

fotmob = ls.FotMob()
fbref = ls.Fbref()

# **Scrapeo de Fotmob para obtener información de cada tiro al arco**

In [ ]:
uploaded = files.upload() # Seleccionar data/manual/codigos_fotmob.xlsx del proyecto

In [ ]:
datos1 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='copa2021')
datos2 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='liga2021')
datos3 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='copa2022')
datos4 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='liga2022')
datos5 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='liga2023')
datos6 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='copa2023')
datos7 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='copa2024')
datos8 = pd.read_excel('codigos_fotmob.xlsx', sheet_name='liga2024')

codigos_copa2021 = datos1[['codigos']]
codigos_liga2021 = datos2[['codigos']]
codigos_copa2022 = datos3[['codigos']]
codigos_liga2022 = datos4[['codigos']]
codigos_liga2023 = datos5[['codigos']]
codigos_copa2023 = datos6[['codigos']]
codigos_copa2024 = datos7[['codigos']]
codigos_liga2024 = datos8[['codigos']]

In [ ]:
# Colab deja descargar 10 archivos por vez: correr en tandas cambiando solo estos tres valores:
torneo = "liga2023"   # copa2023 | liga2023 | copa2024 | liga2024
desde = 0             # índice del primer partido de la tanda (0 = partido 1)
hasta = 10            # índice final, no incluido

codigos_por_torneo = {
    "copa2023": codigos_copa2023,
    "liga2023": codigos_liga2023,
    "copa2024": codigos_copa2024,
    "liga2024": codigos_liga2024,
}
codigos = codigos_por_torneo[torneo]["codigos"]

for i, codigo in enumerate(codigos.iloc[desde:hasta], start=desde + 1):
    print(f"{torneo} - partido {i}: {codigo}")
    info = fotmob.get_match_shotmap(codigo)
    nombre_archivo = f"fotmob_{torneo}_partido{i}.csv"
    pd.DataFrame(info).to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
    files.download(nombre_archivo)
    time.sleep(10)

# **Scrapeo de fbref para obtener la posición de los jugadores**

In [ ]:
info_copa2023 = fbref.get_all_player_season_stats("Copa de la Liga", "2023", save_csv=True)
info_copa2024 = fbref.get_all_player_season_stats("Copa de la Liga", "2024", save_csv=True)
info_liga2023 = fbref.get_all_player_season_stats("Primera Division Argentina", "2023", save_csv=True)
info_liga2024 = fbref.get_all_player_season_stats("Primera Division Argentina", "2024", save_csv=True)

In [ ]:
info_jugadores_copa2023 = info_copa2023[0]
info_arqueros_copa2023 = info_copa2023[1]

info_jugadores_copa2024 = info_copa2024[0]
info_arqueros_copa2024 = info_copa2024[1]

info_jugadores_liga2023 = info_liga2023[0]
info_arqueros_liga2023 = info_liga2023[1]

info_jugadores_liga2024 = info_liga2024[0]
info_arqueros_liga2024 = info_liga2024[1]

In [ ]:
info_jugadores_copa2023.to_csv('info_jugadores_copa2023.csv', index=False)
info_arqueros_copa2023.to_csv('info_arqueros_copa2023.csv', index=False)

info_jugadores_copa2024.to_csv('info_jugadores_copa2024.csv', index=False)
info_arqueros_copa2024.to_csv('info_arqueros_copa2024.csv', index=False)

info_jugadores_liga2023.to_csv('info_jugadores_liga2023.csv', index=False)
info_arqueros_liga2023.to_csv('info_arqueros_liga2023.csv', index=False)

info_jugadores_liga2024.to_csv('info_jugadores_liga2024.csv', index=False)
info_arqueros_liga2024.to_csv('info_arqueros_liga2024.csv', index=False)

files.download('info_jugadores_copa2023.csv')
files.download('info_arqueros_copa2023.csv')

files.download('info_jugadores_copa2024.csv')
files.download('info_arqueros_copa2024.csv')

files.download('info_jugadores_liga2023.csv')
files.download('info_arqueros_liga2023.csv')

files.download('info_jugadores_liga2024.csv')
files.download('info_arqueros_liga2024.csv')

# **Scrapeo de los equipos participantes en cada liga con sus ID de Fotmob**

In [ ]:
# Primero trabajo con las copas de la liga porque las tablas estan divididas en las dos zonas.
tabla_copa2024 = fotmob.get_season_tables("Argentina Copa de la Liga", "2024")
tabla_copa2023 = fotmob.get_season_tables("Argentina Copa de la Liga", "2023")

tabla_A_copa2024 = tabla_copa2024[0]['table']['home']
tabla_B_copa2024 = tabla_copa2024[1]['table']['away']
tabla_A_copa2023 = tabla_copa2023[0]['table']['home']
tabla_B_copa2023 = tabla_copa2023[1]['table']['away']

A_filtrado_copa2024 = [{'name': equipo['name'], 'id': equipo['id']} for equipo in tabla_A_copa2024]
B_filtrado_copa2024 = [{'name': equipo['name'], 'id': equipo['id']} for equipo in tabla_B_copa2024]
A_filtrado_copa2023 = [{'name': equipo['name'], 'id': equipo['id']} for equipo in tabla_A_copa2023]
B_filtrado_copa2023 = [{'name': equipo['name'], 'id': equipo['id']} for equipo in tabla_B_copa2023]

df1_copa2024 = pd.DataFrame(A_filtrado_copa2024)
df2_copa2024 = pd.DataFrame(B_filtrado_copa2024)
df1_copa2023 = pd.DataFrame(A_filtrado_copa2023)
df2_copa2023 = pd.DataFrame(B_filtrado_copa2023)

df_combinado_copa2024 = pd.concat([df1_copa2024, df2_copa2024], ignore_index=True)
df_combinado_copa2023 = pd.concat([df1_copa2023, df2_copa2023], ignore_index=True)

In [ ]:
tabla_liga2024 = fotmob.get_season_tables("Argentina Primera Division", "2024", table = "xg")[["name", "id"]]
tabla_liga2023 = fotmob.get_season_tables("Argentina Primera Division", "2023", table = "xg")[["name", "id"]]

df_liga2024 = pd.DataFrame(tabla_liga2024)
df_liga2023 = pd.DataFrame(tabla_liga2023)

df_liga2024.to_csv('id_equipos_liga2024.csv', index=False, encoding='utf-8-sig')
df_liga2023.to_csv('id_equipos_liga2023.csv', index=False, encoding='utf-8-sig')
df_combinado_copa2024.to_csv('id_equipos_copa2024.csv', index=False, encoding='utf-8-sig')
df_combinado_copa2023.to_csv('id_equipos_copa2023.csv', index=False, encoding='utf-8-sig')

files.download('id_equipos_liga2024.csv')
files.download('id_equipos_liga2023.csv')
files.download('id_equipos_copa2024.csv')
files.download('id_equipos_copa2023.csv')